# Employment Analysis
Monthly workforce snapshots from Jan 2024 – Feb 2026. Used as the denominator for computing true separation rates and establishing pre-DOGE agency baselines.

## Load Data

In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from scripts.data_loader import load_r2_data
import pandas as pd
import numpy as np

### NOTE: This will take half an hour!

Load from Cache or from bucket then cache the DF. 

In [6]:
import pickle

AGG_COLS = [
    "agency_subelement_code",
    "agency_subelement",
    "age_bracket",
    "tenure_code",
    "tenure",
    "appointment_type_code",
    "appointment_type",
    "pay_plan",
    "work_schedule",
    "education_level_bracket",
]

CACHE = Path("employment.pkl")
if CACHE.exists():
    with open(CACHE, "rb") as f:
        e_df = pickle.load(f)
    print("Loaded from cache")
else:
    e_df = load_r2_data("employment", agg_cols=AGG_COLS)
    with open(CACHE, "wb") as f:
        pickle.dump(e_df, f)
    print("Fetched from R2 and cached")

print(f"Shape: {e_df.shape[0]:,} rows x {e_df.shape[1]} columns")
print(f"Months covered: {sorted((e_df['year'].astype(str) + '-' + e_df['month'].astype(str).str.zfill(2)).unique().tolist())}")

  loading employment/2024/employment_202401.txt …
  loading employment/2024/employment_202402.txt …
  loading employment/2024/employment_202403.txt …
  loading employment/2024/employment_202404.txt …
  loading employment/2024/employment_202405.txt …
  loading employment/2024/employment_202406.txt …
  loading employment/2024/employment_202407.txt …
  loading employment/2024/employment_202408.txt …
  loading employment/2024/employment_202409.txt …
  loading employment/2024/employment_202410.txt …
  loading employment/2024/employment_202411.txt …
  loading employment/2024/employment_202412.txt …
  loading employment/2025/employment_202501.txt …
  loading employment/2025/employment_202502.txt …
  loading employment/2025/employment_202503.txt …
  loading employment/2025/employment_202504.txt …
  loading employment/2025/employment_202505.txt …
  loading employment/2025/employment_202506.txt …
  loading employment/2025/employment_202507.txt …
  loading employment/2025/employment_202508.txt …


In [7]:
e_df['ym'] = e_df['year'].astype(str) + '-' + e_df['month'].astype(str).str.zfill(2)
print("Months available:", sorted(e_df['ym'].unique()))

Months available: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02']


## Total Workforce Size Over Time
Sum `count` across all rows for each month to get total headcount.

In [8]:
# Total federal workforce per month
monthly_total = (
    e_df.groupby(['year', 'month', 'ym'])['count']
    .sum()
    .reset_index()
    .sort_values('ym')
)
monthly_total

,year,month,ym,count
0,2024,1,2024-01,2264960
1,2024,2,2024-02,2271498
2,2024,3,2024-03,2278730
3,2024,4,2024-04,2280603
4,2024,5,2024-05,2290825
5,2024,6,2024-06,2300688
6,2024,7,2024-07,2306875
7,2024,8,2024-08,2310761
8,2024,9,2024-09,2313216
9,2024,10,2024-10,2313562


## Agency-Level Headcount Over Time
Track headcount per agency subelement month by month to identify workforce collapse.

In [10]:
# Headcount per agency subelement per month
agency_monthly = (
    e_df.groupby(['ym', 'agency_subelement_code', 'agency_subelement'])['count']
    .sum()
    .reset_index()
    .sort_values(['agency_subelement_code', 'ym'])
)

# Pivot to wide: rows=agency, cols=month
agency_pivot = agency_monthly.pivot_table(
    index=['agency_subelement_code', 'agency_subelement'],
    columns='ym',
    values='count',
    fill_value=0
)
agency_pivot.head(10)

,ym,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,...,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02
agency_subelement_code,agency_subelement,,,,,,,,,,,,,,,,,,,,,
AA00,ADMINISTRATIVE CONFERENCE OF THE UNITED STATES,11.0,11.0,11.0,11.0,11.0,12.0,12.0,12.0,13.0,13.0,...,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0
AB00,AMERICAN BATTLE MONUMENTS COMMISSION,74.0,74.0,74.0,75.0,77.0,78.0,79.0,79.0,78.0,78.0,...,83.0,81.0,80.0,80.0,80.0,78.0,79.0,79.0,77.0,77.0
AF02,AIR FORCE INSPECTION AGENCY (FO),34.0,33.0,33.0,32.0,31.0,31.0,29.0,29.0,30.0,31.0,...,30.0,30.0,29.0,29.0,29.0,27.0,27.0,27.0,27.0,29.0
AF03,AIR FORCE OPERATIONAL TEST AND EVALUATION CENTER,227.0,232.0,232.0,230.0,236.0,239.0,240.0,240.0,242.0,247.0,...,232.0,230.0,228.0,228.0,227.0,202.0,199.0,197.0,194.0,193.0
AF06,AIR FORCE AUDIT AGENCY,499.0,497.0,496.0,492.0,495.0,495.0,493.0,495.0,496.0,494.0,...,485.0,485.0,486.0,483.0,478.0,428.0,426.0,424.0,417.0,416.0
AF07,AIR FORCE OFFICE OF SPECIAL INVESTIGATIONS,1119.0,1123.0,1125.0,1124.0,1127.0,1142.0,1145.0,1141.0,1148.0,1147.0,...,1162.0,1158.0,1151.0,1149.0,1145.0,1105.0,1098.0,1091.0,1076.0,1070.0
AF09,AIR FORCE PERSONNEL CENTER,1501.0,1512.0,1521.0,1544.0,1550.0,1553.0,1569.0,1579.0,1590.0,1600.0,...,1623.0,1621.0,1617.0,1613.0,1588.0,1466.0,1446.0,1428.0,1380.0,1377.0
AF0B,U.S. AIR FORCE ACADEMY,1297.0,1296.0,1309.0,1324.0,1343.0,1392.0,1401.0,1343.0,1361.0,1358.0,...,1331.0,1319.0,1306.0,1278.0,1266.0,1146.0,1142.0,1128.0,1087.0,1082.0
AF0D,"U.S. AIR FORCES, EUROPE",1653.0,1645.0,1646.0,1658.0,1655.0,1709.0,1721.0,1653.0,1657.0,1677.0,...,1655.0,1629.0,1611.0,1574.0,1544.0,1471.0,1454.0,1426.0,1397.0,1386.0


## Pre-DOGE Baseline Workforce Characteristics
Use Jan–Jun 2025 as the pre-DOGE baseline (before executive orders took effect in practice).
Capture agency-level profile: size, age mix, tenure mix, % temporary appointments.

In [11]:
# Pre-DOGE snapshot: average of Jan-Jun 2025
pre_doge = e_df[(e_df['year'] == 2025) & (e_df['month'] <= 6)].copy()

# Total headcount per agency in pre-DOGE period (average monthly)
pre_doge_size = (
    pre_doge.groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={'count': 'avg_headcount_pre_doge'})
)
pre_doge_size.sort_values('avg_headcount_pre_doge', ascending=False).head(20)

,agency_subelement_code,agency_subelement,avg_headcount_pre_doge
521,VATA,VETERANS HEALTH ADMINISTRATION,419028.0
475,TR93,INTERNAL REVENUE SERVICE,99226.0
19,AF1M,AIR FORCE MATERIEL COMMAND,71483.0
350,HSBD,CUSTOMS AND BORDER PROTECTION,66845.0
349,HSBC,TRANSPORTATION SECURITY ADMINISTRATION,66118.0
458,SZ00,SOCIAL SECURITY ADMINISTRATION,54993.0
462,TD03,FEDERAL AVIATION ADMINISTRATION,46202.0
215,DD83,MILITARY TREATMENT FACILITIES UNDER DHA,43808.0
105,ARCE,U.S. ARMY CORPS OF ENGINEERS,38141.0
425,NV24,NAVAL SEA SYSTEMS COMMAND,37196.0


In [12]:
# % young workers (20-29) per agency in pre-DOGE period
pre_doge_total = pre_doge.groupby('agency_subelement_code')['count'].sum().rename('total')

young = (
    pre_doge[pre_doge['age_bracket'].isin(['20-24', '25-29'])]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('young_count')
)

pct_young = (young / pre_doge_total * 100).round(1).reset_index()
pct_young.columns = ['agency_subelement_code', 'pct_young_pre_doge']
pct_young.sort_values('pct_young_pre_doge', ascending=False).head(20)

,agency_subelement_code,pct_young_pre_doge
21,AF1Y,61.6
163,CE00,44.2
356,HT00,37.9
232,DLCA,36.4
53,AF6N,35.8
59,AFZG,29.6
147,ARXK,26.9
414,NS00,25.3
525,ZS00,24.7
123,ARNG,24.6


In [13]:
# % non-tenured (tenure_code 0 = no tenure, 2 = probationary) per agency pre-DOGE
# Tenure codes: 1=permanent career, 2=career conditional (probationary), 3=term/temp, 0=excepted indefinite
non_tenured = (
    pre_doge[pre_doge['tenure_code'].isin(['2', '3', 2.0, 3.0])]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('non_tenured_count')
)

pct_non_tenured = (non_tenured / pre_doge_total * 100).round(1).reset_index()
pct_non_tenured.columns = ['agency_subelement_code', 'pct_non_tenured_pre_doge']
pct_non_tenured.sort_values('pct_non_tenured_pre_doge', ascending=False).head(20)

,agency_subelement_code,pct_non_tenured_pre_doge
437,OP00,100.0
327,HE31,100.0
358,HU05,100.0
523,WU00,100.0
181,DA00,100.0
444,RJ00,100.0
439,PU00,99.8
371,HW00,92.5
414,NS00,92.3
180,CX00,88.2


In [14]:
# % temporary/term appointments (appointment_type_code 10=career, 15=career cond, 20=temp, 30=excepted, etc.)
# Temporary = codes that aren't career or career-conditional
temp_codes = ['20', '30', '38', '50', '55']  # temp, excepted, vet recruitment, etc.

temp_appt = (
    pre_doge[pre_doge['appointment_type_code'].isin(temp_codes)]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('temp_count')
)

pct_temp = (temp_appt / pre_doge_total * 100).round(1).reset_index()
pct_temp.columns = ['agency_subelement_code', 'pct_temp_pre_doge']
pct_temp.sort_values('pct_temp_pre_doge', ascending=False).head(20)

,agency_subelement_code,pct_temp_pre_doge
294,GE00,100.0
53,AF6N,100.0
35,AF38,100.0
158,BK00,100.0
253,DQ00,100.0
327,HE31,100.0
429,NV41,99.8
41,AF40,99.8
349,HSBC,99.7
29,AF2L,99.6


In [15]:
# Build combined pre-DOGE agency profile
agency_profile = (
    pre_doge_size
    .merge(pct_young, on='agency_subelement_code', how='left')
    .merge(pct_non_tenured, on='agency_subelement_code', how='left')
    .merge(pct_temp, on='agency_subelement_code', how='left')
)

# Limit to agencies with meaningful size (>=100 avg headcount)
agency_profile = agency_profile[agency_profile['avg_headcount_pre_doge'] >= 100]
agency_profile.sort_values('avg_headcount_pre_doge', ascending=False).head(20)

,agency_subelement_code,agency_subelement,avg_headcount_pre_doge,pct_young_pre_doge,pct_non_tenured_pre_doge,pct_temp_pre_doge
521,VATA,VETERANS HEALTH ADMINISTRATION,419028.0,5.2,9.7,71.1
475,TR93,INTERNAL REVENUE SERVICE,99226.0,11.4,32.6,2.7
19,AF1M,AIR FORCE MATERIEL COMMAND,71483.0,11.6,19.8,3.3
350,HSBD,CUSTOMS AND BORDER PROTECTION,66845.0,9.8,17.6,3.1
349,HSBC,TRANSPORTATION SECURITY ADMINISTRATION,66118.0,18.8,12.0,99.7
458,SZ00,SOCIAL SECURITY ADMINISTRATION,54993.0,4.1,14.4,6.7
462,TD03,FEDERAL AVIATION ADMINISTRATION,46202.0,9.2,7.0,98.1
215,DD83,MILITARY TREATMENT FACILITIES UNDER DHA,43808.0,3.8,27.6,1.0
105,ARCE,U.S. ARMY CORPS OF ENGINEERS,38141.0,10.9,22.8,7.5
425,NV24,NAVAL SEA SYSTEMS COMMAND,37196.0,15.2,19.5,1.9


## Net Headcount Change: Pre-DOGE vs Now
Compare Jan–Jun 2025 average headcount to Jan–Feb 2026 average to identify agencies with workforce collapse.

In [16]:
# Post-DOGE headcount: Jan-Feb 2026
post_doge = e_df[(e_df['year'] == 2026)].copy()

post_doge_size = (
    post_doge.groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={'count': 'avg_headcount_2026'})
)

# Join and compute change
headcount_change = pre_doge_size.merge(
    post_doge_size[['agency_subelement_code', 'avg_headcount_2026']],
    on='agency_subelement_code',
    how='inner'
)

headcount_change['abs_change'] = headcount_change['avg_headcount_2026'] - headcount_change['avg_headcount_pre_doge']
headcount_change['pct_change'] = (headcount_change['abs_change'] / headcount_change['avg_headcount_pre_doge'] * 100).round(1)

# Largest absolute losses
headcount_change.sort_values('abs_change').head(25)

,agency_subelement_code,agency_subelement,avg_headcount_pre_doge,avg_headcount_2026,abs_change,pct_change
465,TR93,INTERNAL REVENUE SERVICE,99226.0,74252.0,-24974.0,-25.2
508,VATA,VETERANS HEALTH ADMINISTRATION,419028.0,400862.0,-18166.0,-4.3
66,AG11,FOREST SERVICE,33652.0,26340.0,-7312.0,-21.7
19,AF1M,AIR FORCE MATERIEL COMMAND,71483.0,64634.0,-6849.0,-9.6
448,SZ00,SOCIAL SECURITY ADMINISTRATION,54993.0,49782.0,-5211.0,-9.5
91,AM00,U.S. AGENCY FOR INTERNATIONAL DEVELOPMENT,4831.0,256.0,-4575.0,-94.7
374,IN10,NATIONAL PARK SERVICE,18494.0,14076.0,-4418.0,-23.9
348,HSCB,FEDERAL EMERGENCY MANAGEMENT AGENCY,25593.0,21518.0,-4075.0,-15.9
327,HE36,FOOD AND DRUG ADMINISTRATION,20240.0,16428.0,-3812.0,-18.8
329,HE38,NATIONAL INSTITUTES OF HEALTH,20658.0,16858.0,-3800.0,-18.4


In [17]:
# Largest % losses (agencies with >=500 pre-DOGE headcount to filter noise)
headcount_change[
    headcount_change['avg_headcount_pre_doge'] >= 500
].sort_values('pct_change').head(25)

,agency_subelement_code,agency_subelement,avg_headcount_pre_doge,avg_headcount_2026,abs_change,pct_change
91,AM00,U.S. AGENCY FOR INTERNATIONAL DEVELOPMENT,4831.0,256.0,-4575.0,-94.7
199,DD48,DEFENSE HUMAN RESOURCES ACTIVITY,2814.0,1018.0,-1796.0,-63.8
264,EDEN,FEDERAL STUDENT AID,1354.0,744.0,-610.0,-45.1
384,KS00,CORPORATION FOR NATIONAL AND COMMUNITY SERVICE,704.0,406.0,-298.0,-42.3
296,GS03,PUBLIC BUILDINGS SERVICE,5486.0,3235.0,-2251.0,-41.0
255,EDEC,OFFICE FOR CIVIL RIGHTS,559.0,339.0,-220.0,-39.4
323,HE32,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES ADM...,885.0,539.0,-346.0,-39.1
471,TRFD,BUREAU OF THE FISCAL SERVICE,3229.0,2019.0,-1210.0,-37.5
299,GS11,OFFICE OF THE CHIEF FINANCIAL OFFICER,782.0,492.0,-290.0,-37.1
356,HUDD,ASSISTANT SECRETARY FOR COMMUNITY PLANNING AND...,689.0,436.0,-253.0,-36.7


## Separation Rates: Joining Separations to Employment
Compute monthly involuntary separation rate per agency = involuntary separations / beginning-of-month headcount.

In [18]:
# Load separations data
s_df = load_r2_data("separations")

INVOLUNTARY = {"SH", "SJ"}
s_df["sep_class"] = s_df["separation_category_code"].apply(
    lambda c: "involuntary" if c in INVOLUNTARY else ("voluntary" if c in {"SC","SD","SE","SG","SA","SB"} else "other")
)
s_df['ym'] = s_df['year'].astype(str) + '-' + s_df['month'].astype(str).str.zfill(2)

print(f"Separations loaded: {s_df.shape[0]:,} rows")

/Users/destin/Projects/federal-doge-study/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


  loading separations/2024/separations_202401.txt …
  loading separations/2024/separations_202402.txt …
  loading separations/2024/separations_202403.txt …
  loading separations/2024/separations_202404.txt …
  loading separations/2024/separations_202405.txt …
  loading separations/2024/separations_202406.txt …
  loading separations/2024/separations_202407.txt …
  loading separations/2024/separations_202408.txt …
  loading separations/2024/separations_202409.txt …
  loading separations/2024/separations_202410.txt …
  loading separations/2024/separations_202411.txt …
  loading separations/2024/separations_202412.txt …
  loading separations/2025/separations_202501.txt …
  loading separations/2025/separations_202502.txt …
  loading separations/2025/separations_202503.txt …
  loading separations/2025/separations_202504.txt …
  loading separations/2025/separations_202505.txt …
  loading separations/2025/separations_202506.txt …
  loading separations/2025/separations_202507.txt …
  loading se

In [19]:
# Involuntary separations per agency per month
inv_by_agency_month = (
    s_df[s_df['sep_class'] == 'involuntary']
    .groupby(['ym', 'agency_subelement_code'])['count']
    .sum()
    .reset_index()
    .rename(columns={'count': 'inv_sep_count'})
)

# Employment headcount per agency per month
emp_by_agency_month = (
    e_df.groupby(['ym', 'agency_subelement_code'])['count']
    .sum()
    .reset_index()
    .rename(columns={'count': 'headcount'})
)

# Join
rates = emp_by_agency_month.merge(inv_by_agency_month, on=['ym', 'agency_subelement_code'], how='left')
rates['inv_sep_count'] = rates['inv_sep_count'].fillna(0)
rates['inv_sep_rate_pct'] = (rates['inv_sep_count'] / rates['headcount'] * 100).round(3)

print(f"Rate table shape: {rates.shape}")
rates.head()

Rate table shape: (13723, 5)


,ym,agency_subelement_code,headcount,inv_sep_count,inv_sep_rate_pct
0,2024-01,AA00,11,0.0,0.0
1,2024-01,AB00,74,0.0,0.0
2,2024-01,AF02,34,0.0,0.0
3,2024-01,AF03,227,0.0,0.0
4,2024-01,AF06,499,0.0,0.0


In [20]:
# Top agencies by peak involuntary separation rate during DOGE period (Jul 2025 - Feb 2026)
doge_rates = rates[rates['ym'] >= '2025-07'].copy()

peak_rates = (
    doge_rates.groupby('agency_subelement_code')
    .agg(
        max_inv_rate=('inv_sep_rate_pct', 'max'),
        total_inv_seps=('inv_sep_count', 'sum'),
        avg_headcount=('headcount', 'mean')
    )
    .reset_index()
)

# Add agency name
agency_names = e_df[['agency_subelement_code', 'agency_subelement']].drop_duplicates()
peak_rates = peak_rates.merge(agency_names, on='agency_subelement_code', how='left')

peak_rates[
    peak_rates['avg_headcount'] >= 500
].sort_values('max_inv_rate', ascending=False).head(20)

,agency_subelement_code,max_inv_rate,total_inv_seps,avg_headcount,agency_subelement
207,DD48,118.110,1527.0,2096.000,DEFENSE HUMAN RESOURCES ACTIVITY
93,AM00,114.441,3740.0,848.250,U.S. AGENCY FOR INTERNATIONAL DEVELOPMENT
273,EDEN,30.973,282.0,857.250,FEDERAL STUDENT AID
336,HE32,25.733,184.0,572.875,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES ADM...
338,HE34,17.228,414.0,1805.125,HEALTH RESOURCES AND SERVICES ADMINISTRATION
347,HE90,16.468,297.0,1593.125,ADMINISTRATION FOR CHILDREN AND FAMILIES
390,IN10,10.118,4672.0,17162.625,NATIONAL PARK SERVICE
134,ARSE,9.539,139.0,1188.875,HQDA FIELD OPERATING AGENCIES AND STAFF SUPPOR...
340,HE36,7.074,1544.0,16940.125,FOOD AND DRUG ADMINISTRATION
331,HE10,7.052,260.0,2449.125,OFFICE OF THE SECRETARY OF HEALTH AND HUMAN SE...


In [21]:
# Monthly involuntary rate for top DOGE-affected agencies
top_doge_agencies = [
    'AM00',  # USAID
    'HE36',  # FDA
    'HE38',  # NIH
    'HE39',  # CDC
    'ST00',  # State Dept
    'DD48',  # Defense Human Resources Activity
    'SB00',  # Small Business Administration
    'HE34',  # HRSA
]

top_rates = (
    rates[rates['agency_subelement_code'].isin(top_doge_agencies)]
    .merge(agency_names, on='agency_subelement_code', how='left')
    .pivot_table(index='ym', columns='agency_subelement', values='inv_sep_rate_pct', fill_value=0)
    .sort_index()
)
top_rates

agency_subelement,CENTERS FOR DISEASE CONTROL AND PREVENTION,DEFENSE HUMAN RESOURCES ACTIVITY,DEPARTMENT OF STATE,FOOD AND DRUG ADMINISTRATION,HEALTH RESOURCES AND SERVICES ADMINISTRATION,NATIONAL INSTITUTES OF HEALTH,SMALL BUSINESS ADMINISTRATION,U.S. AGENCY FOR INTERNATIONAL DEVELOPMENT
ym,,,,,,,,
2024-01,0.120,0.216,0.164,0.049,0.038,0.122,0.673,0.194
2024-02,0.072,0.144,0.084,0.073,0.115,0.024,0.540,0.021
2024-03,0.024,0.180,0.824,0.063,0.191,0.131,0.126,0.021
2024-04,0.040,0.108,0.727,0.029,0.000,0.083,0.341,0.043
2024-05,0.024,0.036,0.166,0.029,0.000,0.106,0.343,0.043
2024-06,0.212,0.824,0.211,0.048,0.151,0.372,0.226,0.589
2024-07,0.251,0.071,1.056,0.068,0.113,0.139,0.837,0.021
2024-08,0.149,0.035,1.552,0.029,0.189,0.048,0.229,0.250
2024-09,0.429,0.316,0.377,0.062,0.298,0.100,1.635,0.604


## Workforce Profile Shift Over Time
Are agencies getting older, more tenured, or less tenured over time? Track composition changes.

In [22]:
# % young workers (20-29) across entire federal workforce per month
monthly_young = (
    e_df.groupby(['ym', 'age_bracket'])['count']
    .sum()
    .reset_index()
)
monthly_total_map = monthly_total.set_index('ym')['count'].to_dict()
monthly_young['total'] = monthly_young['ym'].map(monthly_total_map)
monthly_young['pct'] = (monthly_young['count'] / monthly_young['total'] * 100).round(2)

young_trend = (
    monthly_young[monthly_young['age_bracket'].isin(['20-24', '25-29'])]
    .groupby('ym')['pct']
    .sum()
    .reset_index()
    .rename(columns={'pct': 'pct_young_20_29'})
    .sort_values('ym')
)
young_trend

,ym,pct_young_20_29
0,2024-01,8.41
1,2024-02,8.44
2,2024-03,8.46
3,2024-04,8.47
4,2024-05,8.64
5,2024-06,8.96
6,2024-07,9.03
7,2024-08,8.91
8,2024-09,8.87
9,2024-10,8.80


In [23]:
# % career (tenure_code 1) vs non-career across entire workforce per month
monthly_tenure = (
    e_df.groupby(['ym', 'tenure_code'])['count']
    .sum()
    .reset_index()
)
monthly_tenure['total'] = monthly_tenure['ym'].map(monthly_total_map)
monthly_tenure['pct'] = (monthly_tenure['count'] / monthly_tenure['total'] * 100).round(2)

tenure_trend = (
    monthly_tenure
    .pivot_table(index='ym', columns='tenure_code', values='pct', fill_value=0)
    .sort_index()
)
tenure_trend

tenure_code,*,0,1,2,3
ym,,,,,
2024-01,0.0,2.77,75.15,18.30,3.79
2024-02,0.0,2.76,75.10,18.36,3.78
2024-03,0.0,2.74,75.11,18.39,3.76
2024-04,0.0,2.75,75.16,18.34,3.75
2024-05,0.0,2.96,74.98,18.27,3.78
2024-06,0.0,3.26,74.65,18.31,3.78
2024-07,0.0,3.25,74.63,18.33,3.79
2024-08,0.0,3.09,74.90,18.24,3.77
2024-09,0.0,2.99,75.02,18.21,3.78
